In [ ]:
import yaml 

# Load the credentials YAML file
credential_path = '../credentials.yml' 
cfg = yaml.load(open(credential_path, "r"), Loader=yaml.Loader)
cfg

In [ ]:
import boto3
import awswrangler as wr

# Credentials
aws_access_key_id = cfg['aws']['aws_access_key_id']
aws_secret_access_key = cfg['aws']['aws_secret_access_key']

# S3 Location
S3_BUCKET = 'cm-aws-s3-data-source'
S3_FOLDER_PATH = 'organization/sales'

# create a authorized session using boto3 + credentials
session = boto3.Session(
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
)
# create s3 client to interact with AWS S3
s3 = session.client(service_name='s3')

Full Load - Snapshot

In [ ]:
# HINTS
# 1. list_objects_v2 >> List all objects
# 2. download_file >> download file
# 3. Download all files to "destination/sales/snapshot"

In [ ]:
# List all objects
response = s3.list_objects_v2(
    Bucket=S3_BUCKET,
    Prefix=S3_FOLDER_PATH
)
response

In [ ]:
import os

file_names = []
contents = response.get('Contents')

for result in contents:
    key = result.get("Key")
    if key[-1] == "/":
        continue
    else:
        file_names.append(key)

# Define local destination path for downloaded files
destination_path = 'destination/sales/snapshot'
os.makedirs(destination_path, exist_ok=True)

for file_name in file_names:
    base_name = os.path.basename(file_name)
    local_file_path = os.path.join(destination_path, base_name)
    s3.download_file(
        Bucket=S3_BUCKET,
        Key=file_name,
        Filename=str(local_file_path)
    )

Full Load - Partition by Ingestion Date

In [ ]:
# HINTS
# 1. list_objects_v2 >> List all objects
# 2. download_file >> download file
# 3. Download all files to "destination/sales/<ingestion_date>"

In [ ]:
from datetime import datetime

file_names = []
contents = response.get('Contents')

for result in contents:
    key = result.get("Key")
    print(key)
    if key[-1] == "/":
        continue
    else:
        file_names.append(key)

# Define local destination path for downloaded files
ingestion_date = datetime.today().strftime('%Y-%m-%d')
destination_path = f'destination/sales/{ingestion_date}'
os.makedirs(destination_path, exist_ok=True)

for file_name in file_names:
    base_name = os.path.basename(file_name)
    local_file_path = os.path.join(destination_path, base_name)
    s3.download_file(
        Bucket=S3_BUCKET,
        Key=file_name,
        Filename=str(local_file_path)
    )

In [ ]:
import os
import glob

# Upload local files to S3 destination bucket
ingestion_date = datetime.today().strftime('%Y-%m-%d')

# S3 destination bucket
s3_destination_bucket = 'cm-aws-s3-destination'
s3_destination_path = f'k1/lybui/sales/{ingestion_date}/'

# Local file path
local_file_path = f'destination/sales/{ingestion_date}'
file_names = glob.glob(f'{local_file_path}/*.csv')

for file_name in file_names:
    base_name = os.path.basename(file_name)
    s3_file_name = s3_destination_path + base_name
    print(s3_file_name)

    s3.upload_file(
        Filename=str(file_name),
        Bucket=s3_destination_bucket,
        Key=s3_file_name
    )

    print(f"Uploaded {file_name} to S3 destination bucket!")